# Learning to Design City-scale Transit Routes
### End-to-End Deep Reinforcement Learning for the Transit Route Network Design Problem (TRNDP)
*Replication and Implementation of **Poudel & Li (arXiv:2512.19767, Dec 2025)***

---

## 📌 Executive Summary
Designing efficient transit routes across a city is an **NP-hard combinatorial optimization problem** with an astronomical search space ($\approx 9.5 \times 10^{115}$ possible combinations on Bloomington, Indiana). This implementation features:

- **Network Topology:** Real-world Bloomington, IN network with $n = 143$ nodes (intersections/stops) and $m = 243$ bidirectional road segments.
- **Design Constraints:** Exactly $K = 16$ routes, each with length $L_{\min} \le |r_k| \le L_{\max} = 14$ nodes, simple self-avoiding walks, bidirectional bus operation.
- **Two-Level Reward Formulation:**
  - **Intermediate Topological Reward ($|r_k| < L_{\max}$):**
    $$R_{\text{partial}} = \beta_0 \Psi - \beta_1 \omega - \beta_2 \left(1 - \frac{|r_k|}{L_{\max}}\right)$$
    where $\Psi$ is the demand coverage potential (fraction of total OD demand reachable) and $\omega$ is route overlap depth across shared edges.
  - **Terminal Simulation Reward ($|r_k| = L_{\max}$):**
    $$R_{\text{final}} = \beta_3 \Psi + \beta_4 \sigma - \beta_5 \tau - \beta_6 \omega$$
    evaluated via mesoscopic traffic simulation (**UXsim**) with service rate $\sigma = N_{\text{boarded}}/N_{\text{want}}$ and normalized travel time $\tau$.
- **Deep RL Architecture:** Shared 4-layer **GATv2** graph attention encoder with edge features (length, free-flow speed), a pointer-style actor head with candidate masking, and a pooled critic head trained via **PPO**.
- **Kaggle Session & Quota Protections:** 100% offline local dataset caching, checkpoint saving/resuming, and time-guarded graceful exit.

---
### 📑 Notebook Workflow:
1. **System & GPU Environment Diagnostics**
2. **Modular Architecture Definitions** (`network_data.py`, `transit_assignment.py`, `bus_dispatcher.py`, `reward.py`, `env.py`, `models.py`, `ppo_agent.py`, `train.py`, `visualize.py`)
3. **Offline Dataset Preparation** (Bloomington CSV/JSON Caching)
4. **Baseline Network Exploration & Interactive Visualizations**
5. **Route Details & Fleet Sizing Analysis**
6. **Training Pipeline with Checkpointing & Resumption**
7. **Performance Comparison Dashboard (Table I Replication)**


In [ ]:
# datasets library no longer needed - data is downloaded via direct HTTPS

In [2]:
!pip install -q torch-geometric gymnasium uxsim plotly matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 24.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 87.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 86.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 MB 28.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 281.0/281.0 kB 20.6 MB/s eta 0:00:00


In [3]:
# 1. System & GPU Environment Diagnostics
import sys
import platform
import torch

print("=" * 70)
print("  SYSTEM & HARDWARE ENVIRONMENT DIAGNOSTICS")
print("=" * 70)
print(f"  OS / Platform      : {platform.system()} {platform.release()} ({platform.machine()})")
print(f"  Python Version     : {sys.version.split()[0]}")
print(f"  PyTorch Version    : {torch.__version__}")
print(f"  CUDA Available     : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    cuda_version = torch.version.cuda
    print(f"  GPU Device         : {gpu_name}")
    print(f"  Total VRAM         : {vram_gb:.2f} GB")
    print(f"  CUDA Version       : {cuda_version}")
else:
    print("  Running on         : CPU Mode (Note: Kaggle CPU provides unlimited hours & 4 vCPUs)")

try:
    import torch_geometric
    print(f"  PyTorch Geometric  : {torch_geometric.__version__}")
except ImportError:
    print("  PyTorch Geometric  : Not loaded")

try:
    import uxsim
    print(f"  UXsim Version      : {getattr(uxsim, '__version__', 'Installed')}")
except ImportError:
    print("  UXsim              : Installed")

print("=" * 70)


  SYSTEM & HARDWARE ENVIRONMENT DIAGNOSTICS
  OS / Platform      : Linux 6.12.90+ (x86_64)
  Python Version     : 3.12.13
  PyTorch Version    : 2.10.0+cu128
  CUDA Available     : True
  GPU Device         : Tesla T4
  Total VRAM         : 14.56 GB
  CUDA Version       : 12.8
  PyTorch Geometric  : 2.8.0.post1
  UXsim Version      : 1.14.2


In [4]:
%%writefile network_data.py
# paste network_data.py contents here

"""
Own from-scratch data loading + UXsim World construction for the
Bloomington TRNDP dataset.

Uses ONLY vanilla UXsim (pip package) primitives -- no code from
AlphaTransit's rl/env.py or vendored uxsim/BusHandler/ is imported here.
The raw CSV/JSON files themselves are just data (same files distributed
on HuggingFace as matrix-multiply/bloomington-tndp).
"""
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from uxsim import World


@dataclass
class NetworkData:
    node_ids: List[str]                 # ordered list, index position == node index used elsewhere
    node_xy: Dict[str, Tuple[float, float]]
    edges: List[Tuple[str, str, float, float]]  # (u, v, length_m, free_flow_speed_mps)
    demand: np.ndarray                  # (n, n) trips/hour, Dij
    existing_routes: Dict[str, List[str]]  # route name -> ordered list of node ids


def _parse_from_frames(nodes_df: pd.DataFrame, links_df: pd.DataFrame, demand_df: pd.DataFrame, routes_records: list) -> NetworkData:
    """
    Shared parsing logic once nodes/links/demand are pandas DataFrames and
    routes_records is a list of dicts with keys "name" and "nodes"
    (a list of node ids) -- works whether the frames came from local CSVs
    or from `datasets.Dataset.to_pandas()` / row iteration over a
    HuggingFace dataset split.
    """
    node_ids = nodes_df["name"].astype(str).tolist()
    node_xy = {
        str(name): (float(x), float(y))
        for name, x, y in zip(nodes_df["name"], nodes_df["x"], nodes_df["y"])
    }

    edges = []
    for u, v, length, speed in zip(links_df["start"], links_df["end"], links_df["length"], links_df["free_flow_speed"]):
        edges.append((str(u), str(v), float(length), float(speed)))

    n = len(node_ids)
    idx = {nid: i for i, nid in enumerate(node_ids)}
    D = np.zeros((n, n), dtype=np.float64)
    for o, d, volume in zip(demand_df["orig"], demand_df["dest"], demand_df["volume"]):
        o, d = str(o), str(d)
        if o in idx and d in idx:
            D[idx[o], idx[d]] += float(volume)

    existing_routes = {}
    for route in routes_records:
        existing_routes[str(route["name"])] = [str(v) for v in route["nodes"]]

    return NetworkData(node_ids=node_ids, node_xy=node_xy, edges=edges, demand=D, existing_routes=existing_routes)


def load_network_data(nodes_csv: str, links_csv: str, demand_csv: str, routes_json: str) -> NetworkData:
    """
    Loads from local CSV/JSON files.
    Actual schema (verified against bloomington_*_standard.csv):
      nodes:  name, x, y
      links:  name, start, end, length (meters), free_flow_speed (m/s)
      demand: orig, dest, volume (trips/hour)
      routes: JSON list of {"name", "short_name", "nodes": [int, ...]}
    """
    nodes_df = pd.read_csv(nodes_csv)
    links_df = pd.read_csv(links_csv)
    demand_df = pd.read_csv(demand_csv)

    import json as _json
    with open(routes_json) as f:
        routes_records = _json.load(f)

    return _parse_from_frames(nodes_df, links_df, demand_df, routes_records)


def load_network_data_from_hf(nodes_ds, links_ds, demand_ds, routes_ds) -> NetworkData:
    """
    Loads directly from HuggingFace `datasets.Dataset` splits, e.g.:

        from datasets import load_dataset
        nodes  = load_dataset("matrix-multiply/bloomington-tndp", "nodes", split="benchmark")
        links  = load_dataset("matrix-multiply/bloomington-tndp", "links", split="benchmark")
        demand = load_dataset("matrix-multiply/bloomington-tndp", "demand", split="benchmark")
        routes = load_dataset("matrix-multiply/bloomington-tndp", "existing_routes", split="benchmark")
        net = load_network_data_from_hf(nodes, links, demand, routes)

    NOTE: this could not be tested against the live dataset from this
    build environment (huggingface.co is not reachable from the sandbox
    used to build/verify the rest of this project). Column names are
    assumed identical to the CSV/JSON schema (same underlying files per
    the AlphaTransit repo's data table), but if HF packaging renamed any
    column, this will raise a KeyError -- run the inspection snippet
    below first to confirm before relying on this in a long training run.

        print(nodes.column_names, nodes[0])
        print(links.column_names, links[0])
        print(demand.column_names, demand[0])
        print(routes.column_names, routes[0])
    """
    nodes_df = nodes_ds.to_pandas()
    links_df = links_ds.to_pandas()
    demand_df = demand_ds.to_pandas()
    routes_records = list(routes_ds)  # list of dicts, one per route

    return _parse_from_frames(nodes_df, links_df, demand_df, routes_records)


def build_world(net: NetworkData, tmax: float = 10000.0, deltan: int = 5) -> World:
    """
    Builds a vanilla UXsim World from the raw network -- no bus-specific
    logic here, just nodes/links matching the paper's Sec. IV-A setup
    (DELTAN=5 platoons, DELTAT=1s reaction time, Tmax=10,000s).
    """
    W = World(
        name="bloomington_trndp",
        deltan=deltan,
        tmax=tmax,
        print_mode=0,
        save_mode=0,
        show_mode=0,
        random_seed=0,
    )
    for nid in net.node_ids:
        x, y = net.node_xy[nid]
        W.addNode(nid, x, y)

    for (u, v, length, speed) in net.edges:
        W.addLink(f"{u}_{v}", u, v, length=length, free_flow_speed=speed)
        W.addLink(f"{v}_{u}", v, u, length=length, free_flow_speed=speed)  # bidirectional per Def. 2

    return W


def inject_car_demand(W: World, net: NetworkData, alpha: float, t_end: float = 3600.0):
    """
    Injects the (1 - alpha) share of OD demand as private-car trips, so
    buses experience real congestion from competing car traffic -- this
    is the piece that a free-flow-only approximation would skip.
    """
    n = len(net.node_ids)
    car_share = 1.0 - alpha
    for i in range(n):
        for j in range(n):
            trips_per_hour = net.demand[i, j] * car_share
            if trips_per_hour <= 0:
                continue
            W.adddemand(net.node_ids[i], net.node_ids[j], t_start=0, t_end=t_end, flow=trips_per_hour / 3600.0)

Writing network_data.py


In [5]:
%%writefile transit_assignment.py
# paste transit_assignment.py contents here

"""
Own from-scratch transit demand assignment + Frequency-of-Service (FOS)
computation, matching Definition 4 and Equation 19 of the paper.

ASSUMPTION FLAGGED HONESTLY: the paper does not fully specify the exact
demand-assignment algorithm used to estimate segment loads before FOS
sizing -- it only cites "max-load principle [7],[68]" and states loads
are normalized by the number of overlapping routes per segment. This
module implements a standard min-transfers-then-min-time shortest path
assignment over a route-layered graph, which is a reasonable, well-known
approach for this kind of static transit assignment, but it is our own
design choice for the unspecified part, not a byte-exact reproduction of
unpublished internal code.

Stop spacing s_k = 1 for all routes (confirmed in paper's supplementary
hyperparameter table), so every route node is a stop -- Definition 4
reduces to S(r_k) = all nodes in r_k.
"""
from __future__ import annotations

import heapq
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np

from network_data import NetworkData

TRANSFER_PENALTY_SECONDS = 5 * 60.0  # heuristic transfer disutility, see module docstring
DELTA_MAX = 0.8                       # comfort threshold (Eq. 19)
BUS_CAPACITY = 40                     # C_k (Sec. IV-A)


@dataclass
class RouteGeometry:
    name: str
    nodes: List[str]                       # stop sequence (s_k=1 -> == route node sequence)
    segment_time_s: List[float]            # travel time for each consecutive stop pair (both directions equal, undirected edge reused)


def build_route_geometries(net: NetworkData, routes: Dict[str, List[str]]) -> Dict[str, RouteGeometry]:
    """Computes per-segment free-flow travel time along each route's exact edges."""
    edge_time = {}
    for (u, v, length, speed) in net.edges:
        t = length / max(speed, 1e-6)
        edge_time[(u, v)] = t
        edge_time[(v, u)] = t  # bidirectional (Def. 2)

    geometries = {}
    for name, nodes in routes.items():
        seg_times = []
        for a, b in zip(nodes[:-1], nodes[1:]):
            if (a, b) not in edge_time:
                raise ValueError(f"Route {name}: no edge between consecutive stops {a}->{b}; route is not a valid simple path on G.")
            seg_times.append(edge_time[(a, b)])
        geometries[name] = RouteGeometry(name=name, nodes=nodes, segment_time_s=seg_times)
    return geometries


def _build_assignment_graph(geometries: Dict[str, RouteGeometry]):
    """
    Layered graph over states (node, route_or_None):
      - "route" state (p, k): currently riding route k, physically at stop p
      - "street" state (p, None): not on any vehicle, physically at stop p
    Edges:
      - board:    (p, None) -> (p, k)      weight 0,               for every route k stopping at p
      - alight:   (p, k)    -> (p, None)   weight 0
      - transfer: (p, k1)   -> (p, k2)     weight TRANSFER_PENALTY, k1 != k2, both stop at p
      - ride:     (p, k)    -> (q, k)      weight = segment travel time, for consecutive stops p,q on route k (both directions)
    Cost is a single scalar (transfer penalty dominates, so shortest path naturally minimizes transfers first in practice).
    """
    adjacency: Dict[Tuple[str, str], List[Tuple[Tuple[str, str], float]]] = {}

    def add_edge(u_state, v_state, w):
        adjacency.setdefault(u_state, []).append((v_state, w))

    stops_to_routes: Dict[str, List[str]] = {}
    for k, geo in geometries.items():
        for p in geo.nodes:
            stops_to_routes.setdefault(p, []).append(k)

    for p, route_list in stops_to_routes.items():
        street_state = (p, "__street__")
        for k in route_list:
            ride_state = (p, k)
            add_edge(street_state, ride_state, 0.0)   # board
            add_edge(ride_state, street_state, 0.0)   # alight
        for k1 in route_list:
            for k2 in route_list:
                if k1 != k2:
                    add_edge((p, k1), (p, k2), TRANSFER_PENALTY_SECONDS)  # transfer

    for k, geo in geometries.items():
        for i in range(len(geo.nodes) - 1):
            p, q = geo.nodes[i], geo.nodes[i + 1]
            t = geo.segment_time_s[i]
            add_edge((p, k), (q, k), t)  # ride forward
            add_edge((q, k), (p, k), t)  # ride backward (bidirectional operation, Def. 2)

    return adjacency, stops_to_routes


def _dijkstra(adjacency, source_state):
    dist = {source_state: 0.0}
    prev = {}
    pq = [(0.0, source_state)]
    visited = set()
    while pq:
        d, u = heapq.heappop(pq)
        if u in visited:
            continue
        visited.add(u)
        for v, w in adjacency.get(u, []):
            nd = d + w
            if v not in dist or nd < dist[v]:
                dist[v] = nd
                prev[v] = u
                heapq.heappush(pq, (nd, v))
    return dist, prev


def _reconstruct_route_segments_used(prev, source_state, target_state) -> List[Tuple[str, str, str]]:
    """Walk back the shortest path, returning (route, from_node, to_node) for every 'ride' edge used."""
    path_states = [target_state]
    u = target_state
    while u != source_state:
        u = prev[u]
        path_states.append(u)
    path_states.reverse()

    segments = []
    for a, b in zip(path_states[:-1], path_states[1:]):
        (p_a, k_a), (p_b, k_b) = a, b
        if k_a == k_b and k_a != "__street__" and p_a != p_b:
            segments.append((k_a, p_a, p_b))
    return segments


def assign_transit_demand(
    net: NetworkData, geometries: Dict[str, RouteGeometry], alpha: float
) -> Dict[str, Dict[Tuple[str, str], float]]:
    """
    Assigns D^transit = alpha * D onto route segments via min-cost
    (transfer-penalized) shortest path per OD pair.

    Returns: {route_name: {(from_node, to_node): assigned_load_trips_per_hour}}
    """
    adjacency, stops_to_routes = _build_assignment_graph(geometries)
    served_nodes = set(stops_to_routes.keys())

    segment_load: Dict[str, Dict[Tuple[str, str], float]] = {k: {} for k in geometries}

    n = len(net.node_ids)
    # Group by unique origins actually served, to avoid O(n) Dijkstra calls when demand is sparse
    origins_with_demand = [i for i in range(n) if net.demand[i, :].sum() > 0 and net.node_ids[i] in served_nodes]

    for i in origins_with_demand:
        orig = net.node_ids[i]
        source_state = (orig, "__street__")
        dist, prev = _dijkstra(adjacency, source_state)

        for j in range(n):
            demand_ij = net.demand[i, j] * alpha
            if demand_ij <= 0:
                continue
            dest = net.node_ids[j]
            target_state = (dest, "__street__")
            if target_state not in dist:
                continue  # unreachable via current transit network -- unserved demand

            segments = _reconstruct_route_segments_used(prev, source_state, target_state)
            for (route_name, a, b) in segments:
                key = (a, b) if (a, b) in segment_load[route_name] or (b, a) not in segment_load[route_name] else (b, a)
                segment_load[route_name][key] = segment_load[route_name].get(key, 0.0) + demand_ij

    return segment_load


def _collapse_into_legs(segments: List[Tuple[str, str, str]]) -> List[Tuple[str, str, str]]:
    """
    Collapses a sequence of consecutive (route, from, to) hops into
    (route, board_stop, alight_stop) legs, merging consecutive hops on the
    same route into one ride, so a transfer is only counted where the
    route actually changes.
    """
    if not segments:
        return []
    legs = []
    cur_route, board_stop, _ = segments[0]
    last_to = segments[0][2]
    for route, a, b in segments[1:]:
        if route == cur_route:
            last_to = b
        else:
            legs.append((cur_route, board_stop, last_to))
            cur_route, board_stop, last_to = route, a, b
    legs.append((cur_route, board_stop, last_to))
    return legs


def compute_passenger_itineraries(
    net: NetworkData, geometries: Dict[str, RouteGeometry], alpha: float
) -> Dict[Tuple[str, str], List[Tuple[str, str, str]]]:
    """
    Returns {(orig_node, dest_node): [(route, board_stop, alight_stop), ...]}
    for every OD pair with positive transit demand and a feasible transit
    path. Reuses the same shortest-path assignment as
    `assign_transit_demand` (kept separate to avoid recomputation coupling
    -- this is intentionally a second, independent Dijkstra pass per OD
    pair since itinerary reconstruction needs the raw segment list, not
    just aggregated loads).
    """
    adjacency, stops_to_routes = _build_assignment_graph(geometries)
    served_nodes = set(stops_to_routes.keys())
    n = len(net.node_ids)

    itineraries: Dict[Tuple[str, str], List[Tuple[str, str, str]]] = {}
    origins_with_demand = [i for i in range(n) if net.demand[i, :].sum() > 0 and net.node_ids[i] in served_nodes]

    for i in origins_with_demand:
        orig = net.node_ids[i]
        source_state = (orig, "__street__")
        dist, prev = _dijkstra(adjacency, source_state)
        for j in range(n):
            if net.demand[i, j] * alpha <= 0:
                continue
            dest = net.node_ids[j]
            if orig == dest or dest not in served_nodes:
                continue
            target_state = (dest, "__street__")
            if target_state not in dist:
                continue
            segments = _reconstruct_route_segments_used(prev, source_state, target_state)
            legs = _collapse_into_legs(segments)
            if legs:
                itineraries[(orig, dest)] = legs

    return itineraries


def compute_frequencies(
    geometries: Dict[str, RouteGeometry],
    segment_load: Dict[str, Dict[Tuple[str, str], float]],
) -> Dict[str, int]:
    """
    Equation 19: F_k = ceil( Q_k,max^norm / (delta_max * C_k) ).

    Overlap normalization: for each segment, divide its load by the number
    of routes that also serve that same physical edge, before taking the
    max over route k's own segments (see module docstring re: this being
    our own reasonable interpretation of the paper's stated normalization).
    """
    # Count how many routes cover each undirected physical edge
    edge_route_count: Dict[Tuple[str, str], int] = {}
    for k, geo in geometries.items():
        seen_edges_this_route = set()
        for a, b in zip(geo.nodes[:-1], geo.nodes[1:]):
            edge_key = tuple(sorted((a, b)))
            if edge_key not in seen_edges_this_route:
                edge_route_count[edge_key] = edge_route_count.get(edge_key, 0) + 1
                seen_edges_this_route.add(edge_key)

    frequencies = {}
    for k, geo in geometries.items():
        loads = segment_load.get(k, {})
        max_norm_load = 0.0
        for (a, b), load in loads.items():
            edge_key = tuple(sorted((a, b)))
            overlap = max(edge_route_count.get(edge_key, 1), 1)
            norm_load = load / overlap
            max_norm_load = max(max_norm_load, norm_load)
        f_k = int(np.ceil(max_norm_load / (DELTA_MAX * BUS_CAPACITY))) if max_norm_load > 0 else 1
        frequencies[k] = max(f_k, 1)  # at least 1 bus/hour even for zero-demand routes
    return frequencies

Writing transit_assignment.py


In [6]:
%%writefile passenger_demand.py
# paste passenger_demand.py contents here

"""
Own from-scratch passenger generation: turns OD itineraries + hourly
demand rates into individual synthetic passengers with Poisson arrival
times over the simulation horizon.
"""
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Dict, List, Tuple

import numpy as np

from network_data import NetworkData


@dataclass
class Passenger:
    name: int
    orig: str
    dest: str
    depart_time: float                       # time they arrive at the first stop, wanting to travel
    itinerary: List[Tuple[str, str, str]]     # [(route, board_stop, alight_stop), ...]
    leg_idx: int = 0
    wait_start_time: float = None             # set when they start waiting for the *current* leg's bus
    board_time: float = None                  # most recent boarding time (reset per leg)
    n_transfers: int = 0
    completed: bool = False
    aborted: bool = False                     # could not board within sim horizon
    total_wait_time: float = 0.0
    total_invehicle_time: float = 0.0

    def current_leg(self):
        if self.leg_idx < len(self.itinerary):
            return self.itinerary[self.leg_idx]
        return None


def generate_passengers(
    net: NetworkData,
    itineraries: Dict[Tuple[str, str], List[Tuple[str, str, str]]],
    alpha: float,
    t_horizon_s: float,
    rng: np.random.Generator,
) -> List[Passenger]:
    """
    For each OD pair with a known itinerary, generates a Poisson process
    of passenger arrivals at rate (alpha * D_ij) trips/hour over
    t_horizon_s seconds.
    """
    idx = {nid: i for i, nid in enumerate(net.node_ids)}
    passengers: List[Passenger] = []
    pid = 0
    horizon_hours = t_horizon_s / 3600.0

    for (orig, dest), legs in itineraries.items():
        i, j = idx[orig], idx[dest]
        rate_per_hour = net.demand[i, j] * alpha
        if rate_per_hour <= 0:
            continue
        expected_count = rate_per_hour * horizon_hours
        n_passengers = rng.poisson(expected_count)
        if n_passengers == 0:
            continue
        depart_times = np.sort(rng.uniform(0, t_horizon_s, size=n_passengers))
        for t in depart_times:
            passengers.append(
                Passenger(name=pid, orig=orig, dest=dest, depart_time=float(t), itinerary=list(legs))
            )
            pid += 1

    passengers.sort(key=lambda p: p.depart_time)
    return passengers


Writing passenger_demand.py


In [7]:
%%writefile bus_dispatcher.py
# paste bus_dispatcher.py contents here

"""
Own from-scratch bus simulation layer on top of VANILLA UXsim.

Core technique ("leg-chaining"): each bus trip is broken into one UXsim
Vehicle per route segment (mode="single_trip", route forced via
Vehicle.enforce_route so it follows the RL-designed edges exactly, not
UXsim's own route choice). When a leg's vehicle reaches its destination
node, a `node_event` callback (the same mechanism vanilla UXsim's
TaxiHandler uses for pickup/dropoff) runs our boarding/alighting logic
and then dynamically spawns the *next* leg's vehicle with
departure_time = arrival_time + DWELL_TIME_S, producing a bus that
dwells at every stop and is subject to real congestion from the
car population sharing the same links.

None of this imports rl/env.py or the vendored uxsim/BusHandler module.
"""
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Dict, List, Tuple

from uxsim import World

from transit_assignment import RouteGeometry, BUS_CAPACITY
from passenger_demand import Passenger

DWELL_TIME_S = 60.0  # Sec. IV-A


@dataclass
class SimStats:
    n_boarded: int = 0
    n_want: int = 0
    n_completed: int = 0
    n_transfer_trips: int = 0
    wait_times: List[float] = field(default_factory=list)
    travel_times: List[float] = field(default_factory=list)   # wait + in-vehicle, per boarded passenger (so-far if incomplete)
    invehicle_times: List[float] = field(default_factory=list)


class BusDispatcher:
    def __init__(self, W: World, geometries: Dict[str, RouteGeometry], frequencies: Dict[str, int], capacity: int = BUS_CAPACITY):
        self.W = W
        self.geometries = geometries
        self.frequencies = frequencies
        self.capacity = capacity

        self.waiting_queues: Dict[Tuple[str, str], List[Passenger]] = {}   # (stop, route) -> [Passenger]
        self.bus_load: Dict[str, int] = {}
        self.bus_passengers: Dict[str, List[Passenger]] = {}

        self.pending_passengers: List[Passenger] = []  # sorted by depart_time, consumed via pointer
        self._pending_ptr = 0

        self.stats = SimStats()
        self._bus_counter = 0

    def set_passengers(self, passengers: List[Passenger]):
        self.pending_passengers = sorted(passengers, key=lambda p: p.depart_time)
        self._pending_ptr = 0
        self.stats.n_want = len(passengers)

    def _release_ready(self, now: float):
        """Move passengers whose depart_time has arrived into their first-leg waiting queue."""
        n = len(self.pending_passengers)
        while self._pending_ptr < n and self.pending_passengers[self._pending_ptr].depart_time <= now:
            p = self.pending_passengers[self._pending_ptr]
            leg = p.current_leg()
            if leg is not None:
                route, board_stop, _ = leg
                p.wait_start_time = now
                self.waiting_queues.setdefault((board_stop, route), []).append(p)
            self._pending_ptr += 1

    def schedule_all_initial_departures(self, t_horizon_s: float):
        """Kicks off the very first leg of every scheduled bus trip, both directions, for every route."""
        for route_name, geo in self.geometries.items():
            freq = self.frequencies.get(route_name, 1)
            headway_s = 3600.0 / max(freq, 1)

            for direction_nodes in (geo.nodes, list(reversed(geo.nodes))):
                if len(direction_nodes) < 2:
                    continue
                t = 0.0
                while t < t_horizon_s:
                    self._bus_counter += 1
                    bus_id = f"bus_{route_name}_{self._bus_counter}"
                    self.bus_load[bus_id] = 0
                    self.bus_passengers[bus_id] = []
                    self._dispatch_leg(route_name, direction_nodes, next_stop_idx=1, departure_time=t, bus_id=bus_id)
                    t += headway_s

    def _dispatch_leg(self, route_name: str, direction_nodes: List[str], next_stop_idx: int, departure_time: float, bus_id: str):
        """Creates the UXsim vehicle for the leg ending at direction_nodes[next_stop_idx]."""
        orig = direction_nodes[next_stop_idx - 1]
        dest = direction_nodes[next_stop_idx]
        link_name = f"{orig}_{dest}"

        veh = self.W.addVehicle(
            orig, dest, departure_time,
            name=f"{bus_id}_leg{next_stop_idx}",
            mode="single_trip",
        )
        veh.enforce_route([link_name], set_avoid=True)
        veh.node_event[self.W.get_node(dest)] = lambda: self._on_arrival(
            route_name, direction_nodes, next_stop_idx, bus_id
        )

    def _on_arrival(self, route_name: str, direction_nodes: List[str], stop_idx: int, bus_id: str):
        now = self.W.TIME * self.W.DELTAT
        stop = direction_nodes[stop_idx]
        self._release_ready(now)

        # --- Alighting ---
        for p in self.bus_passengers[bus_id][:]:
            leg = p.current_leg()
            if leg is not None and leg[2] == stop:
                self.bus_passengers[bus_id].remove(p)
                self.bus_load[bus_id] -= 1
                p.total_invehicle_time += now - p.board_time
                p.leg_idx += 1
                next_leg = p.current_leg()
                if next_leg is None:
                    p.completed = True
                    self.stats.n_completed += 1
                    total_time = p.total_wait_time + p.total_invehicle_time
                    self.stats.travel_times.append(total_time)
                else:
                    p.n_transfers += 1
                    self.stats.n_transfer_trips += 1
                    p.wait_start_time = now
                    next_route, next_board_stop, _ = next_leg
                    self.waiting_queues.setdefault((next_board_stop, next_route), []).append(p)

        # --- Boarding ---
        queue = self.waiting_queues.get((stop, route_name), [])
        capacity_left = self.capacity - self.bus_load[bus_id]
        boarding = queue[:max(capacity_left, 0)]
        remaining = queue[max(capacity_left, 0):]
        self.waiting_queues[(stop, route_name)] = remaining

        for p in boarding:
            wait = now - (p.wait_start_time if p.wait_start_time is not None else p.depart_time)
            p.total_wait_time += wait
            self.stats.wait_times.append(wait)
            p.board_time = now
            if not getattr(p, "_ever_boarded", False):
                p._ever_boarded = True
                self.stats.n_boarded += 1
            self.bus_passengers[bus_id].append(p)
            self.bus_load[bus_id] += 1

        # --- Chain next leg, or end of line ---
        if stop_idx < len(direction_nodes) - 1:
            self._dispatch_leg(route_name, direction_nodes, stop_idx + 1, now + DWELL_TIME_S, bus_id)
        # else: terminus reached; any remaining onboard passengers indicate an
        # itinerary/route mismatch (shouldn't happen if itineraries were built
        # from this same route's geometry).

    def finalize_incomplete(self):
        """
        At simulation end, passengers still riding or still waiting count
        toward Nboarded's average travel time using their so-far
        accumulated time (matches paper's 'mid-journey at simulation end'
        handling for the Travel Time metric, Sec. IV-C).

        Uses the ACTUAL final simulated clock time (W.TIME * W.DELTAT)
        rather than the nominal horizon passed to schedule_all_initial_departures,
        since UXsim continues simulating already-dispatched vehicles until
        they complete their trips, which can run past the nominal horizon.
        """
        t_end = self.W.TIME * self.W.DELTAT
        for bus_id, riders in self.bus_passengers.items():
            for p in riders:
                so_far_invehicle = max(t_end - p.board_time, 0.0)
                total_time = p.total_wait_time + p.total_invehicle_time + so_far_invehicle
                self.stats.travel_times.append(total_time)
        # Passengers still waiting (never boarded) at sim end are correctly
        # excluded from Nboarded-based averages (wait_times/travel_times)
        # by construction -- they were never appended to those lists.


Writing bus_dispatcher.py


In [8]:
%%writefile reward.py
# paste reward.py contents here

"""
Own from-scratch reward computation, matching the paper's Eq. 11-15 exactly.

    Psi   (Eq. 11): coverage potential = demand on OD pairs reachable via
                     the currently-built network, over total demand.
    omega          : route overlap -- average, over every network edge
                     covered by >=1 route, of (count_covering - 1)/(K-1).
    R_partial (Eq. 12): 40*Psi - 20*omega, minus an under-length penalty
                     15*(1 - |r_k|/L_max) if a route terminates early.
    sigma (Eq. 13): N_boarded / N_want
    tau   (Eq. 14): min( mean(travel_time_p)/3600 over boarded p, 1 )
    R_final (Eq. 15): 30*Psi + 15*sigma - 15*tau - 10*omega
"""
from __future__ import annotations

from typing import Dict, List

import numpy as np

from network_data import NetworkData
from transit_assignment import RouteGeometry, _build_assignment_graph, _dijkstra
from bus_dispatcher import SimStats

BETA0, BETA1, BETA2 = 40.0, 20.0, 15.0   # R_partial coefficients (Eq. 12)
BETA3, BETA4, BETA5, BETA6 = 30.0, 15.0, 15.0, 10.0  # R_final coefficients (Eq. 15)


def compute_psi(net: NetworkData, geometries: Dict[str, RouteGeometry]) -> float:
    """
    Eq. 11: fraction of TOTAL demand (not modal-split-scaled) whose OD
    pair is reachable via some path through the currently-built network
    (with transfers allowed, same layered graph as transit_assignment).
    """
    total_demand = net.demand.sum()
    if total_demand <= 0 or not geometries:
        return 0.0

    adjacency, stops_to_routes = _build_assignment_graph(geometries)
    served_nodes = set(stops_to_routes.keys())
    idx = {nid: i for i, nid in enumerate(net.node_ids)}

    reachable_demand = 0.0
    n = len(net.node_ids)
    origins_with_demand = [i for i in range(n) if net.demand[i, :].sum() > 0 and net.node_ids[i] in served_nodes]

    for i in origins_with_demand:
        orig = net.node_ids[i]
        dist, _ = _dijkstra(adjacency, (orig, "__street__"))
        for j in range(n):
            if net.demand[i, j] <= 0:
                continue
            dest = net.node_ids[j]
            if orig == dest:
                continue
            if (dest, "__street__") in dist:
                reachable_demand += net.demand[i, j]

    return float(reachable_demand / total_demand)


def compute_omega(geometries: Dict[str, RouteGeometry], num_routes_total: int) -> float:
    """
    Route overlap: for every physical edge covered by >=1 route, depth =
    (count_covering - 1) / (K - 1), K = num_routes_total (the configured
    total route budget, e.g. 16 -- NOTE: the paper's text is ambiguous on
    whether K here means the fixed total design capacity or the number
    of routes built so far; we use the fixed total capacity as the more
    natural reading of "shared by all routes", and flag this explicitly
    as our own interpretation of an underspecified detail.)
    """
    if num_routes_total <= 1:
        return 0.0

    edge_route_count: Dict[tuple, int] = {}
    for k, geo in geometries.items():
        seen = set()
        for a, b in zip(geo.nodes[:-1], geo.nodes[1:]):
            key = tuple(sorted((a, b)))
            if key not in seen:
                edge_route_count[key] = edge_route_count.get(key, 0) + 1
                seen.add(key)

    if not edge_route_count:
        return 0.0

    depths = [(count - 1) / (num_routes_total - 1) for count in edge_route_count.values()]
    return float(np.mean(depths))


def compute_partial_reward(
    psi: float, omega: float, route_terminated_early: bool = False, route_len: int = 0, max_len: int = 14
) -> float:
    r = BETA0 * psi - BETA1 * omega
    if route_terminated_early:
        r -= BETA2 * (1 - route_len / max_len)
    return r


def compute_final_reward(psi: float, omega: float, stats: SimStats) -> Dict[str, float]:
    sigma = stats.n_boarded / stats.n_want if stats.n_want > 0 else 0.0
    if stats.n_boarded > 0 and stats.travel_times:
        mean_travel_hours = float(np.mean(stats.travel_times)) / 3600.0
        tau = min(mean_travel_hours, 1.0)
    else:
        tau = 1.0  # no one boarded -> worst-case travel time signal
    reward = BETA3 * psi + BETA4 * sigma - BETA5 * tau - BETA6 * omega
    return {"reward": reward, "psi": psi, "omega": omega, "sigma": sigma, "tau": tau}


Writing reward.py


In [9]:
%%writefile env.py
# paste env.py contents here

"""
Own from-scratch Gymnasium environment for the TRNDP, assembling:
  network_data.py, transit_assignment.py, passenger_demand.py,
  bus_dispatcher.py, reward.py

State features (16-dim per node) follow the paper's own equations
(2)-(8) plus flags -- NOT reverse-engineered from rl/env.py:

  0,1   : normalized (x, y)
  2     : normalized node degree
  3,4   : d_out(i), d_in(i)                          -- Eq. 2 (OD marginals, whole network)
  5,6   : a_cand_{i->cur}, a_cand_{i<-cur}            -- Eq. 3,4 (gated: nonzero only if i in Ct)
  7,8   : a_core_{i->core}, a_core_{i<-core}          -- Eq. 5,6 (gated: nonzero only if i in Vcore = Vcur u Vcmp)
  9,10  : a_all_{i->cur}, a_all_{i<-cur}              -- Eq. 7 (ungated, w.r.t. Vcur)
  11,12 : a_all_{i->cmp}, a_all_{i<-cmp}              -- Eq. 8 (ungated, w.r.t. Vcmp)
  13    : 1{i in Vcur}
  14    : fraction of COMPLETED routes so far that contain i, in [0,1]
  15    : 1{i in Ct} (valid next node)

DESIGN CHOICES made where the paper's text is silent (flagged honestly):
  - Vcore in Eq. 5/6 is read as Vcur u Vcmp ("any designed route" so far,
    including the in-progress one), since Eq. 7/8 already separately
    distinguish Vcur vs Vcmp -- Vcore reads as the union.
  - Action index `n_nodes` (NO_VALID_ACTION) doubles as a voluntary
    "terminate route early" action once the route has >=2 nodes, and is
    the forced action when the candidate set Ct is empty.
  - Since our own policy (models.py) hard-masks logits to -inf for
    invalid actions, this env does not need an "agent picked an illegal
    action" recovery path in normal training -- it trusts the action is
    always in the current valid mask, matching how models.py samples.
"""
from __future__ import annotations

import random
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Set, Tuple

import gymnasium as gym
import numpy as np
from gymnasium import spaces

from network_data import NetworkData, load_network_data, load_network_data_from_hf, build_world, inject_car_demand
from transit_assignment import (
    build_route_geometries, assign_transit_demand, compute_frequencies, compute_passenger_itineraries,
)
from passenger_demand import generate_passengers
from bus_dispatcher import BusDispatcher
from reward import compute_psi, compute_omega, compute_partial_reward, compute_final_reward

N_NODE_FEATURES = 16
NO_VALID_ACTION_LABEL = "__terminate__"


class TransitEnv(gym.Env):
    def __init__(
        self,
        net: NetworkData,
        num_routes: int = 16,
        max_route_length: int = 14,
        alpha: float = 0.3,
        t_sim: float = 10000.0,
        seed: int = 0,
    ):
        super().__init__()
        self.net = net
        self.num_routes = num_routes
        self.max_route_length = max_route_length
        self.alpha = alpha
        self.t_sim = t_sim
        self.rng = random.Random(seed)
        self.np_rng = np.random.default_rng(seed)

        self.n_nodes = len(self.net.node_ids)
        self.node_idx = {nid: i for i, nid in enumerate(self.net.node_ids)}
        self._build_static_graph()

        self.action_space = spaces.Discrete(self.n_nodes + 1)
        self.observation_space = spaces.Dict(
            {
                "node_features": spaces.Box(low=-np.inf, high=np.inf, shape=(self.n_nodes, N_NODE_FEATURES), dtype=np.float32),
                "edge_index": spaces.Box(low=0, high=self.n_nodes, shape=(2, self.edge_index.shape[1]), dtype=np.int64),
                "edge_features": spaces.Box(low=-np.inf, high=np.inf, shape=(self.edge_features.shape[0], 2), dtype=np.float32),
                "route_progress": spaces.Box(low=0, high=1, shape=(self.num_routes,), dtype=np.float32),
                "frontier_index": spaces.Discrete(self.n_nodes + 1),
            }
        )

        # Precompute the whole-network OD marginals (Eq. 2) -- static across the episode
        self._d_out_all = self.net.demand.sum(axis=1)
        self._d_in_all = self.net.demand.sum(axis=0)

        self._reset_episode_state()

    @classmethod
    def from_csv(cls, nodes_csv: str, links_csv: str, demand_csv: str, routes_json: str, **kwargs) -> "TransitEnv":
        net = load_network_data(nodes_csv, links_csv, demand_csv, routes_json)
        return cls(net, **kwargs)

    @classmethod
    def from_huggingface(cls, nodes_ds, links_ds, demand_ds, routes_ds, **kwargs) -> "TransitEnv":
        """
        Example:
            from datasets import load_dataset
            nodes  = load_dataset("matrix-multiply/bloomington-tndp", "nodes", split="benchmark")
            links  = load_dataset("matrix-multiply/bloomington-tndp", "links", split="benchmark")
            demand = load_dataset("matrix-multiply/bloomington-tndp", "demand", split="benchmark")
            routes = load_dataset("matrix-multiply/bloomington-tndp", "existing_routes", split="benchmark")
            env = TransitEnv.from_huggingface(nodes, links, demand, routes, num_routes=16, max_route_length=14)
        """
        net = load_network_data_from_hf(nodes_ds, links_ds, demand_ds, routes_ds)
        return cls(net, **kwargs)

    # ---------------------------------------------------------------- graph
    def _build_static_graph(self):
        out_neighbors: Dict[int, List[int]] = {i: [] for i in range(self.n_nodes)}
        edges_u, edges_v, lengths, speeds = [], [], [], []
        for (u, v, length, speed) in self.net.edges:
            ui, vi = self.node_idx[u], self.node_idx[v]
            for a, b in [(ui, vi), (vi, ui)]:
                out_neighbors[a].append(b)
                edges_u.append(a)
                edges_v.append(b)
                lengths.append(length)
                speeds.append(speed)

        self.out_neighbors = out_neighbors
        self.edge_index = np.array([edges_u, edges_v], dtype=np.int64)
        lengths = np.array(lengths, dtype=np.float32)
        speeds = np.array(speeds, dtype=np.float32)
        self.edge_features = np.stack(
            [lengths / max(lengths.max(), 1e-6), speeds / max(speeds.max(), 1e-6)], axis=1
        ).astype(np.float32)

        xs = np.array([self.net.node_xy[nid][0] for nid in self.net.node_ids], dtype=np.float32)
        ys = np.array([self.net.node_xy[nid][1] for nid in self.net.node_ids], dtype=np.float32)
        self._x_norm = (xs - xs.min()) / max(xs.max() - xs.min(), 1e-6)
        self._y_norm = (ys - ys.min()) / max(ys.max() - ys.min(), 1e-6)
        degrees = np.array([len(out_neighbors[i]) for i in range(self.n_nodes)], dtype=np.float32)
        self._degree_norm = degrees / max(degrees.max(), 1e-6)

    # ------------------------------------------------------------- episode
    def _reset_episode_state(self):
        self.completed_routes: Dict[str, List[str]] = {}       # name -> node id list
        self.completed_route_sets: List[Set[str]] = []          # one set per completed route
        self.current_route: List[str] = []
        self.frontier_idx: int = self.n_nodes  # "no frontier" sentinel until first route starts
        self.route_number = 0

    def reset(self, seed: Optional[int] = None, options: Optional[dict] = None):
        if seed is not None:
            self.rng.seed(seed)
            self.np_rng = np.random.default_rng(seed)
        self._reset_episode_state()
        start_node = self.rng.choice(self.net.node_ids)
        self.current_route = [start_node]
        self.frontier_idx = self.node_idx[start_node]
        obs = self._get_obs()
        return obs, {}

    def _candidates(self) -> List[int]:
        if self.frontier_idx == self.n_nodes:
            return []
        visited = set(self.node_idx[n] for n in self.current_route)
        return [nb for nb in self.out_neighbors[self.frontier_idx] if nb not in visited]

    # ------------------------------------------------------------ features
    def _get_obs(self) -> dict:
        Vcur = set(self.node_idx[n] for n in self.current_route)
        Vcmp = set()
        for s in self.completed_route_sets:
            Vcmp |= set(self.node_idx[n] for n in s)
        Vcore = Vcur | Vcmp
        Ct = set(self._candidates())

        n = self.n_nodes
        feats = np.zeros((n, N_NODE_FEATURES), dtype=np.float32)
        feats[:, 0] = self._x_norm
        feats[:, 1] = self._y_norm
        feats[:, 2] = self._degree_norm
        feats[:, 3] = self._d_out_all / max(self._d_out_all.max(), 1e-6)
        feats[:, 4] = self._d_in_all / max(self._d_in_all.max(), 1e-6)

        D = self.net.demand
        Vcur_list = list(Vcur)
        Vcore_list = list(Vcore)

        if Vcur_list:
            d_out_cur = D[:, Vcur_list].sum(axis=1)  # sum_j in Vcur D[i,j]
            d_in_cur = D[Vcur_list, :].sum(axis=0)   # sum_j in Vcur D[j,i]
        else:
            d_out_cur = np.zeros(n)
            d_in_cur = np.zeros(n)

        if Vcore_list:
            d_out_core = D[:, Vcore_list].sum(axis=1)
            d_in_core = D[Vcore_list, :].sum(axis=0)
        else:
            d_out_core = np.zeros(n)
            d_in_core = np.zeros(n)

        Vcmp_list = list(Vcmp)
        if Vcmp_list:
            d_out_cmp = D[:, Vcmp_list].sum(axis=1)
            d_in_cmp = D[Vcmp_list, :].sum(axis=0)
        else:
            d_out_cmp = np.zeros(n)
            d_in_cmp = np.zeros(n)

        denom = max(D.max(), 1e-6) * max(n, 1)  # generic normalizer for demand-sum features

        for i in range(n):
            in_Ct = i in Ct
            in_Vcore = i in Vcore
            feats[i, 5] = (d_out_cur[i] / denom) if in_Ct else 0.0
            feats[i, 6] = (d_in_cur[i] / denom) if in_Ct else 0.0
            feats[i, 7] = (d_out_core[i] / denom) if in_Vcore else 0.0
            feats[i, 8] = (d_in_core[i] / denom) if in_Vcore else 0.0
            feats[i, 9] = d_out_cur[i] / denom
            feats[i, 10] = d_in_cur[i] / denom
            feats[i, 11] = d_out_cmp[i] / denom
            feats[i, 12] = d_in_cmp[i] / denom
            feats[i, 13] = 1.0 if i in Vcur else 0.0
            feats[i, 15] = 1.0 if in_Ct else 0.0

        n_completed = max(len(self.completed_route_sets), 1)
        frac_completed = np.zeros(n, dtype=np.float32)
        for s in self.completed_route_sets:
            for nid in s:
                frac_completed[self.node_idx[nid]] += 1.0
        feats[:, 14] = frac_completed / n_completed

        route_progress = np.zeros(self.num_routes, dtype=np.float32)
        for i in range(min(self.route_number, self.num_routes)):
            route_progress[i] = 1.0
        if self.route_number < self.num_routes:
            route_progress[self.route_number] = len(self.current_route) / self.max_route_length

        return {
            "node_features": feats,
            "edge_index": self.edge_index,
            "edge_features": self.edge_features,
            "route_progress": route_progress,
            "frontier_index": self.frontier_idx,
        }

    # ---------------------------------------------------------------- step
    def _current_geometries(self):
        """All routes (completed so far + current in-progress) as RouteGeometry, for Psi/omega."""
        all_routes = dict(self.completed_routes)
        if len(self.current_route) >= 2:
            all_routes[f"__current_{self.route_number}__"] = self.current_route
        return build_route_geometries(self.net, all_routes)

    def step(self, action: int):
        candidates = self._candidates()
        terminate = action == self.n_nodes or (candidates and action not in candidates)
        # Note: per module docstring, we trust action in-support from our own masked
        # policy; the `action not in candidates` branch is a defensive fallback only.

        if not terminate:
            next_node = self.net.node_ids[action]
            self.current_route.append(next_node)
            self.frontier_idx = action

        route_len = len(self.current_route)
        route_finished = terminate or route_len >= self.max_route_length or not self._candidates()

        if not route_finished:
            geometries = self._current_geometries()
            psi = compute_psi(self.net, geometries)
            omega = compute_omega(geometries, self.num_routes)
            reward = compute_partial_reward(psi, omega)
            obs = self._get_obs()
            return obs, reward, False, False, {"psi": psi, "omega": omega}

        # --- route finished: apply early-termination penalty if applicable, then full sim ---
        early = route_len < self.max_route_length
        geometries = self._current_geometries()
        psi = compute_psi(self.net, geometries)
        omega = compute_omega(geometries, self.num_routes)
        partial_component = compute_partial_reward(
            psi, omega, route_terminated_early=early, route_len=route_len, max_len=self.max_route_length
        )

        # Full traffic simulation over ALL routes completed so far, including this one
        route_name = f"route_{self.route_number}"
        self.completed_routes[route_name] = list(self.current_route)
        self.completed_route_sets.append(set(self.current_route))

        sim_geometries = build_route_geometries(self.net, self.completed_routes)
        segment_load = assign_transit_demand(self.net, sim_geometries, self.alpha)
        frequencies = compute_frequencies(sim_geometries, segment_load)
        itineraries = compute_passenger_itineraries(self.net, sim_geometries, self.alpha)
        passengers = generate_passengers(self.net, itineraries, self.alpha, self.t_sim, self.np_rng)

        W = build_world(self.net, tmax=self.t_sim)
        inject_car_demand(W, self.net, self.alpha, t_end=self.t_sim)
        dispatcher = BusDispatcher(W, sim_geometries, frequencies)
        dispatcher.set_passengers(passengers)
        dispatcher.schedule_all_initial_departures(self.t_sim)
        W.exec_simulation()
        dispatcher.finalize_incomplete()

        final_info = compute_final_reward(psi, omega, dispatcher.stats)
        reward = partial_component + final_info["reward"]

        self.route_number += 1
        episode_done = self.route_number >= self.num_routes

        if not episode_done:
            start_node = self.rng.choice(self.net.node_ids)
            self.current_route = [start_node]
            self.frontier_idx = self.node_idx[start_node]
        else:
            self.current_route = []
            self.frontier_idx = self.n_nodes

        obs = self._get_obs()
        info = {"psi": psi, "omega": omega, **final_info}
        return obs, reward, episode_done, False, info

Writing env.py


In [10]:
%%writefile models.py
# paste models.py contents here

"""
Own from-scratch GATv2 Actor-Critic for the Transit Route Network Design Problem.

Architecture (matches paper's stated design):
  - 4 GATv2 blocks, pre-LayerNorm, residual connections, head-averaging
    (not concatenation) for stable multi-head aggregation across depth.
  - Edge features (length, free_flow_speed) injected at every GAT layer.
  - Pointer-style actor head: scores every node against the current
    "frontier" node embedding (the end of the route being built).
  - Pooled critic head: mean-pooled graph embedding + route-progress vector.

This is intentionally written independently of rl/models.py in the
AlphaTransit repo -- it only *consumes* the TransitEnv observation
contract (rl/env.py), not their network code.
"""
from __future__ import annotations

import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv


@dataclass
class ModelConfig:
    node_feat_dim: int = 16
    edge_feat_dim: int = 2
    hidden_dim: int = 128
    num_heads: int = 4
    num_layers: int = 4
    num_routes: int = 16
    dropout: float = 0.0


class GATv2Block(nn.Module):
    """
    Pre-LN residual GATv2 block with head-averaging.

    h_{l+1} = h_l + GATv2( LN(h_l), edge_index, edge_attr )

    GATv2Conv is configured with concat=False so multi-head outputs are
    averaged (not concatenated), keeping the hidden dimension constant
    across all 4 stacked blocks -- this is what "head-averaging" refers
    to in the paper's architecture description.
    """

    def __init__(self, hidden_dim: int, num_heads: int, edge_dim: int, dropout: float = 0.0):
        super().__init__()
        self.norm = nn.LayerNorm(hidden_dim)
        self.conv = GATv2Conv(
            in_channels=hidden_dim,
            out_channels=hidden_dim,
            heads=num_heads,
            concat=False,          # head-averaging, not concatenation
            edge_dim=edge_dim,
            dropout=dropout,
            add_self_loops=True,
        )
        self.act = nn.GELU()

    def forward(self, h: torch.Tensor, edge_index: torch.Tensor, edge_attr: torch.Tensor) -> torch.Tensor:
        h_norm = self.norm(h)
        out = self.conv(h_norm, edge_index, edge_attr=edge_attr)
        out = self.act(out)
        return h + out  # residual


class GATv2Encoder(nn.Module):
    """Stack of GATv2Blocks producing per-node contextual embeddings."""

    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.input_proj = nn.Linear(cfg.node_feat_dim, cfg.hidden_dim)
        self.blocks = nn.ModuleList(
            [
                GATv2Block(cfg.hidden_dim, cfg.num_heads, cfg.edge_feat_dim, cfg.dropout)
                for _ in range(cfg.num_layers)
            ]
        )
        self.final_norm = nn.LayerNorm(cfg.hidden_dim)

    def forward(self, node_features: torch.Tensor, edge_index: torch.Tensor, edge_attr: torch.Tensor) -> torch.Tensor:
        h = self.input_proj(node_features)
        for block in self.blocks:
            h = block(h, edge_index, edge_attr)
        return self.final_norm(h)


class PointerActorHead(nn.Module):
    """
    Pointer-style actor: for a given frontier node embedding h_f and every
    candidate node embedding h_i, compute a compatibility score

        logit_i = w_a^T tanh(W1 h_f + W2 h_i + W3 p)

    where p is a small projection of the route-progress vector, broadcast
    to every node. This lets the policy condition next-node choice on how
    far along route construction currently is.
    """

    def __init__(self, hidden_dim: int, num_routes: int):
        super().__init__()
        self.w1 = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.w2 = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.progress_proj = nn.Linear(num_routes, hidden_dim, bias=False)
        self.score = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, node_emb: torch.Tensor, frontier_emb: torch.Tensor, route_progress: torch.Tensor) -> torch.Tensor:
        # node_emb: (N, H)  frontier_emb: (H,)  route_progress: (R,)
        p = self.progress_proj(route_progress)  # (H,)
        combined = torch.tanh(self.w1(frontier_emb) + self.w2(node_emb) + p)  # (N, H)
        logits = self.score(combined).squeeze(-1)  # (N,)
        return logits


class PooledCriticHead(nn.Module):
    """V(s) = MLP( mean_pool(node embeddings) || route_progress )."""

    def __init__(self, hidden_dim: int, num_routes: int):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim + num_routes, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, node_emb: torch.Tensor, route_progress: torch.Tensor) -> torch.Tensor:
        graph_emb = node_emb.mean(dim=0)  # (H,)
        x = torch.cat([graph_emb, route_progress], dim=-1)
        return self.mlp(x).squeeze(-1)  # scalar


class GATv2ActorCritic(nn.Module):
    """
    Full actor-critic: shared GATv2 encoder, pointer actor head, pooled
    critic head. A "no-frontier" (NO_VALID_ACTION) node embedding is
    handled by falling back to a learned dummy embedding when
    frontier_index == n_nodes (per TransitEnv's action_space semantics).
    """

    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.encoder = GATv2Encoder(cfg)
        self.actor_head = PointerActorHead(cfg.hidden_dim, cfg.num_routes)
        self.critic_head = PooledCriticHead(cfg.hidden_dim, cfg.num_routes)
        self.no_frontier_embedding = nn.Parameter(torch.randn(cfg.hidden_dim) * 0.02)

    def forward(
        self,
        node_features: torch.Tensor,   # (N, F)
        edge_index: torch.Tensor,      # (2, E) long
        edge_attr: torch.Tensor,       # (E, 2)
        route_progress: torch.Tensor,  # (R,)
        frontier_index: int,           # scalar int, may equal N (no frontier)
        valid_mask: torch.Tensor,      # (N,) bool -- True where action is legal
    ):
        """
        Returns:
            logits: (N+1,) -- last entry is the NO_VALID_ACTION logit,
                     which is forced to -inf unless valid_mask is all-False.
            value:  scalar V(s)
        """
        node_emb = self.encoder(node_features, edge_index, edge_attr)  # (N, H)
        n_nodes = node_emb.shape[0]

        if frontier_index >= n_nodes:
            frontier_emb = self.no_frontier_embedding
        else:
            frontier_emb = node_emb[frontier_index]

        node_logits = self.actor_head(node_emb, frontier_emb, route_progress)  # (N,)

        # Mask invalid actions with -inf (paper's Option-3 masking still
        # keeps is_valid_next as a *feature*; we additionally hard-mask at
        # the policy's output so the agent's action distribution only
        # spans currently-legal moves during rollout/PPO update).
        masked_logits = node_logits.masked_fill(~valid_mask, float("-inf"))

        no_valid_action_available = (~valid_mask).all()
        no_valid_logit = torch.tensor(
            0.0 if no_valid_action_available else float("-inf"),
            device=node_logits.device,
        )

        logits = torch.cat([masked_logits, no_valid_logit.unsqueeze(0)], dim=0)  # (N+1,)
        value = self.critic_head(node_emb, route_progress)
        return logits, value

Writing models.py


In [11]:
%%writefile ppo_agent.py
# paste ppo_agent.py contents here

"""
Own from-scratch PPO implementation (clipped surrogate + GAE) for the
TransitEnv from the AlphaTransit repo (rl/env.py).

This does not reuse rl/ppo_agent.py -- only TransitEnv itself.
"""
from __future__ import annotations

from dataclasses import dataclass, field
from typing import List, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical

from models import GATv2ActorCritic, ModelConfig

# Node feature indices, per rl/env.py's observation_space docstring:
# 0 x, 1 y, 2 degree, 3 d_out, 4 d_in,
# 5 d_out_cur_local, 6 d_in_cur_local, 7 d_out_comp_local, 8 d_in_comp_local,
# 9 d_out_cur_global, 10 d_in_cur_global, 11 d_out_comp_global, 12 d_in_comp_global,
# 13 in_current_route_flag, 14 in_completed_routes, 15 is_valid_next_flag
IS_VALID_NEXT_FEATURE_IDX = 15


@dataclass
class PPOConfig:
    gamma: float = 0.999           # matches paper's --gamma=0.999 (long-horizon route credit assignment)
    gae_lambda: float = 0.95
    clip_eps: float = 0.2
    value_coef: float = 0.5
    entropy_coef: float = 0.01
    lr: float = 3e-4
    epochs_per_update: int = 4
    minibatch_size: int = 32
    max_grad_norm: float = 0.5
    rollout_steps: int = 512
    device: str = "cpu"


@dataclass
class Transition:
    node_features: np.ndarray
    edge_index: np.ndarray
    edge_features: np.ndarray
    route_progress: np.ndarray
    frontier_index: int
    valid_mask: np.ndarray
    action: int
    log_prob: float
    value: float
    reward: float
    done: bool


@dataclass
class RolloutBuffer:
    transitions: List[Transition] = field(default_factory=list)

    def add(self, t: Transition):
        self.transitions.append(t)

    def clear(self):
        self.transitions.clear()

    def __len__(self):
        return len(self.transitions)


def obs_to_valid_mask(obs: dict, n_nodes: int) -> np.ndarray:
    """True where the node's is_valid_next_flag feature == 1."""
    return obs["node_features"][:, IS_VALID_NEXT_FEATURE_IDX] > 0.5


def compute_gae(
    rewards: np.ndarray,
    values: np.ndarray,
    dones: np.ndarray,
    last_value: float,
    gamma: float,
    lam: float,
):
    """
    delta_t   = r_t + gamma * V(s_{t+1}) * (1 - done_t) - V(s_t)
    A_t       = sum_l (gamma*lam)^l * delta_{t+l}
    return_t  = A_t + V(s_t)
    """
    T = len(rewards)
    advantages = np.zeros(T, dtype=np.float32)
    last_gae = 0.0
    next_value = last_value
    for t in reversed(range(T)):
        next_non_terminal = 1.0 - dones[t]
        delta = rewards[t] + gamma * next_value * next_non_terminal - values[t]
        last_gae = delta + gamma * lam * next_non_terminal * last_gae
        advantages[t] = last_gae
        next_value = values[t]
    returns = advantages + values
    return advantages, returns


class PPOAgent:
    def __init__(self, model_cfg: ModelConfig, ppo_cfg: PPOConfig):
        self.ppo_cfg = ppo_cfg
        self.device = torch.device(ppo_cfg.device)
        self.model = GATv2ActorCritic(model_cfg).to(self.device)
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=ppo_cfg.lr)

    def _obs_to_tensors(self, obs: dict):
        node_features = torch.as_tensor(obs["node_features"], dtype=torch.float32, device=self.device)
        edge_index = torch.as_tensor(obs["edge_index"], dtype=torch.long, device=self.device)
        edge_features = torch.as_tensor(obs["edge_features"], dtype=torch.float32, device=self.device)
        route_progress = torch.as_tensor(obs["route_progress"], dtype=torch.float32, device=self.device)
        return node_features, edge_index, edge_features, route_progress

    @torch.no_grad()
    def act(self, obs: dict):
        """Sample an action from the current policy for a single env step."""
        node_features, edge_index, edge_features, route_progress = self._obs_to_tensors(obs)
        frontier_index = int(obs["frontier_index"])
        n_nodes = node_features.shape[0]

        valid_mask_np = obs_to_valid_mask(obs, n_nodes)
        valid_mask = torch.as_tensor(valid_mask_np, dtype=torch.bool, device=self.device)

        logits, value = self.model(
            node_features, edge_index, edge_features, route_progress, frontier_index, valid_mask
        )
        dist = Categorical(logits=logits)
        action = dist.sample()
        log_prob = dist.log_prob(action)
        return int(action.item()), float(log_prob.item()), float(value.item()), valid_mask_np

    def evaluate_actions(
        self,
        node_features_list,
        edge_index_list,
        edge_features_list,
        route_progress_list,
        frontier_indices,
        valid_masks,
        actions,
    ):
        """
        Batched re-evaluation for the PPO update. TransitEnv graphs are all
        the same fixed 143-node Bloomington network, so we process the
        minibatch as a simple Python loop over variable-size per-sample
        graphs (still correct, just not vectorized across a PyG Batch --
        acceptable at this network scale of 143 nodes / 486 edges).
        """
        log_probs, values, entropies = [], [], []
        for i in range(len(actions)):
            logits, value = self.model(
                node_features_list[i],
                edge_index_list[i],
                edge_features_list[i],
                route_progress_list[i],
                frontier_indices[i],
                valid_masks[i],
            )
            dist = Categorical(logits=logits)
            log_probs.append(dist.log_prob(torch.tensor(actions[i], device=self.device)))
            entropies.append(dist.entropy())
            values.append(value)
        return torch.stack(log_probs), torch.stack(values), torch.stack(entropies)

    def update(self, buffer: RolloutBuffer, last_value: float):
        cfg = self.ppo_cfg
        transitions = buffer.transitions
        T = len(transitions)

        rewards = np.array([t.reward for t in transitions], dtype=np.float32)
        values = np.array([t.value for t in transitions], dtype=np.float32)
        dones = np.array([t.done for t in transitions], dtype=np.float32)

        advantages, returns = compute_gae(rewards, values, dones, last_value, cfg.gamma, cfg.gae_lambda)
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        old_log_probs = np.array([t.log_prob for t in transitions], dtype=np.float32)
        actions = np.array([t.action for t in transitions], dtype=np.int64)

        # Pre-convert all per-step tensors once (small network -> fine to keep in memory)
        node_features_t, edge_index_t, edge_features_t, route_progress_t = [], [], [], []
        frontier_t, valid_mask_t = [], []
        for t in transitions:
            nf, ei, ef, rp = self._obs_to_tensors(
                {
                    "node_features": t.node_features,
                    "edge_index": t.edge_index,
                    "edge_features": t.edge_features,
                    "route_progress": t.route_progress,
                }
            )
            node_features_t.append(nf)
            edge_index_t.append(ei)
            edge_features_t.append(ef)
            route_progress_t.append(rp)
            frontier_t.append(t.frontier_index)
            valid_mask_t.append(torch.as_tensor(t.valid_mask, dtype=torch.bool, device=self.device))

        indices = np.arange(T)
        stats = {"policy_loss": 0.0, "value_loss": 0.0, "entropy": 0.0}
        n_updates = 0

        for _ in range(cfg.epochs_per_update):
            np.random.shuffle(indices)
            for start in range(0, T, cfg.minibatch_size):
                mb_idx = indices[start : start + cfg.minibatch_size]
                if len(mb_idx) == 0:
                    continue

                mb_actions = actions[mb_idx]
                mb_old_log_probs = torch.as_tensor(old_log_probs[mb_idx], device=self.device)
                mb_advantages = torch.as_tensor(advantages[mb_idx], device=self.device)
                mb_returns = torch.as_tensor(returns[mb_idx], device=self.device)

                new_log_probs, new_values, entropy = self.evaluate_actions(
                    [node_features_t[i] for i in mb_idx],
                    [edge_index_t[i] for i in mb_idx],
                    [edge_features_t[i] for i in mb_idx],
                    [route_progress_t[i] for i in mb_idx],
                    [frontier_t[i] for i in mb_idx],
                    [valid_mask_t[i] for i in mb_idx],
                    mb_actions,
                )

                ratio = torch.exp(new_log_probs - mb_old_log_probs)
                surr1 = ratio * mb_advantages
                surr2 = torch.clamp(ratio, 1.0 - cfg.clip_eps, 1.0 + cfg.clip_eps) * mb_advantages
                policy_loss = -torch.min(surr1, surr2).mean()

                value_loss = F.mse_loss(new_values, mb_returns)
                entropy_bonus = entropy.mean()

                loss = policy_loss + cfg.value_coef * value_loss - cfg.entropy_coef * entropy_bonus

                self.optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(self.model.parameters(), cfg.max_grad_norm)
                self.optimizer.step()

                stats["policy_loss"] += policy_loss.item()
                stats["value_loss"] += value_loss.item()
                stats["entropy"] += entropy_bonus.item()
                n_updates += 1

        for k in stats:
            stats[k] /= max(n_updates, 1)
        return stats


Writing ppo_agent.py


In [12]:
%%writefile train.py
# paste train.py contents here

"""
Entry point: trains our own GATv2 Actor-Critic + PPO implementation
against the real TransitEnv (rl/env.py) from the AlphaTransit repo,
using the actual Bloomington dataset and UXsim traffic simulator.

Features:
- Robust checkpointing (saves model, optimizer, RNG states, history)
- Resume capability (--resume) to chain multiple Kaggle sessions
- Time guard (--max_hours) to exit cleanly before Kaggle's 12-hour session kill
- Offline-first local data loading (--data_source csv) to eliminate HF Hub warnings
"""
from __future__ import annotations

import argparse
import json
import os
import random
import time

import numpy as np
import torch

from env import TransitEnv  # our own from-scratch environment
from models import ModelConfig
from ppo_agent import PPOAgent, PPOConfig, RolloutBuffer, Transition, obs_to_valid_mask


def parse_args():
    p = argparse.ArgumentParser()
    # Data arguments (defaults to local offline files to avoid HF Hub warnings/rate limits)
    p.add_argument("--data_source", type=str, choices=["csv", "huggingface"], default="csv")
    p.add_argument("--nodes_csv", type=str, default="./data/bloomington_nodes.csv")
    p.add_argument("--links_csv", type=str, default="./data/bloomington_links.csv")
    p.add_argument("--demand_csv", type=str, default="./data/bloomington_demand.csv")
    p.add_argument("--routes_json", type=str, default="./data/bloomington_routes.json")
    p.add_argument("--hf_dataset_name", type=str, default="matrix-multiply/bloomington-tndp")
    p.add_argument("--hf_split", type=str, default="benchmark")

    # Environment & RL hyperparameters
    p.add_argument("--num_routes", type=int, default=16)
    p.add_argument("--max_route_length", type=int, default=14)
    p.add_argument("--alpha", type=float, default=0.3)
    p.add_argument("--t_sim", type=float, default=10000.0)
    p.add_argument("--num_episodes", type=int, default=200)
    p.add_argument("--rollout_steps", type=int, default=256)
    p.add_argument("--hidden_dim", type=int, default=128)
    p.add_argument("--num_heads", type=int, default=4)
    p.add_argument("--num_layers", type=int, default=4)
    p.add_argument("--lr", type=float, default=3e-4)
    p.add_argument("--gamma", type=float, default=0.999)
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--log_every", type=int, default=1)
    p.add_argument("--device", type=str, default="cuda" if torch.cuda.is_available() else "cpu")

    # Checkpointing & Kaggle quota / session management
    p.add_argument("--checkpoint_dir", type=str, default="./checkpoints", help="Directory to save model checkpoints")
    p.add_argument("--save_every", type=int, default=5, help="Save periodic checkpoint every N episodes")
    p.add_argument("--resume", type=str, default=None, help="Path to checkpoint .pt file to resume from")
    p.add_argument(
        "--max_hours",
        type=float,
        default=11.0,
        help="Max hours to run before graceful exit (Kaggle hard-kills at 12 hours)",
    )
    return p.parse_args()


def build_env(args):
    common_kwargs = dict(
        num_routes=args.num_routes,
        max_route_length=args.max_route_length,
        alpha=args.alpha,
        t_sim=args.t_sim,
        seed=args.seed,
    )
    if args.data_source == "huggingface":
        from datasets import load_dataset
        nodes = load_dataset(args.hf_dataset_name, "nodes", split=args.hf_split)
        links = load_dataset(args.hf_dataset_name, "links", split=args.hf_split)
        demand = load_dataset(args.hf_dataset_name, "demand", split=args.hf_split)
        routes = load_dataset(
            "json",
            data_files=f"hf://datasets/{args.hf_dataset_name}/standard/bloomington_existing_routes.json",
            split="train",
        )
        return TransitEnv.from_huggingface(nodes, links, demand, routes, **common_kwargs)
    else:
        assert args.nodes_csv and args.links_csv and args.demand_csv and args.routes_json, (
            "--nodes_csv/--links_csv/--demand_csv/--routes_json are required when --data_source=csv"
        )
        return TransitEnv.from_csv(args.nodes_csv, args.links_csv, args.demand_csv, args.routes_json, **common_kwargs)


def run_episode(env: TransitEnv, agent: PPOAgent, buffer: RolloutBuffer, max_steps: int):
    obs, info = env.reset()
    episode_reward = 0.0
    ep_done = False
    steps = 0

    while not ep_done and steps < max_steps:
        n_nodes = env.n_nodes
        valid_mask_np = obs_to_valid_mask(obs, n_nodes)

        action, log_prob, value, _ = agent.act(obs)
        next_obs, reward, route_done, ep_done_flag, info = env.step(action)
        ep_done = bool(ep_done_flag)

        buffer.add(
            Transition(
                node_features=obs["node_features"].copy(),
                edge_index=obs["edge_index"].copy(),
                edge_features=obs["edge_features"].copy(),
                route_progress=obs["route_progress"].copy(),
                frontier_index=int(obs["frontier_index"]),
                valid_mask=valid_mask_np.copy(),
                action=action,
                log_prob=log_prob,
                value=value,
                reward=reward,
                done=ep_done,
            )
        )
        episode_reward += reward
        obs = next_obs
        steps += 1

    return episode_reward, steps, obs


def save_checkpoint(path: str, episode: int, agent: PPOAgent, ep_reward: float, best_reward: float, history: list, args):
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    rng_state = {
        "torch": torch.get_rng_state(),
        "torch_cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
        "numpy": np.random.get_state(),
        "random": random.getstate(),
    }
    checkpoint = {
        "episode": episode,
        "model_state_dict": agent.model.state_dict(),
        "optimizer_state_dict": agent.optimizer.state_dict(),
        "rng_state": rng_state,
        "last_reward": ep_reward,
        "best_reward": best_reward,
        "history": history,
        "args": vars(args),
    }
    torch.save(checkpoint, path)

    # Save human-readable JSON history log alongside checkpoints
    log_path = os.path.join(os.path.dirname(os.path.abspath(path)), "training_history.json")
    try:
        with open(log_path, "w", encoding="utf-8") as f:
            json.dump(history, f, indent=2)
    except Exception:
        pass


def main():
    args = parse_args()
    torch.manual_seed(args.seed)
    np.random.seed(args.seed)
    random.seed(args.seed)

    env = build_env(args)
    print(
        f"TransitEnv ready: n_nodes={env.n_nodes}, num_routes={env.num_routes}, "
        f"max_route_length={env.max_route_length}"
    )

    model_cfg = ModelConfig(
        node_feat_dim=16,
        edge_feat_dim=2,
        hidden_dim=args.hidden_dim,
        num_heads=args.num_heads,
        num_layers=args.num_layers,
        num_routes=env.num_routes,
    )
    ppo_cfg = PPOConfig(lr=args.lr, gamma=args.gamma, rollout_steps=args.rollout_steps, device=args.device)
    agent = PPOAgent(model_cfg, ppo_cfg)
    print(f"Using device: {ppo_cfg.device}")

    # Checkpoint & Resume initialization
    start_episode = 1
    best_reward = -float("inf")
    history = []

    if args.resume:
        if os.path.isfile(args.resume):
            print(f"Loading checkpoint from: {args.resume}")
            checkpoint = torch.load(args.resume, map_location=agent.device, weights_only=False)
            agent.model.load_state_dict(checkpoint["model_state_dict"])
            agent.optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

            rng = checkpoint.get("rng_state", {})
            if "torch" in rng and rng["torch"] is not None:
                torch.set_rng_state(rng["torch"].cpu())
            if "torch_cuda" in rng and rng["torch_cuda"] is not None and torch.cuda.is_available():
                try:
                    torch.cuda.set_rng_state_all([s.cpu() for s in rng["torch_cuda"]])
                except Exception:
                    pass
            if "numpy" in rng:
                np.random.set_state(rng["numpy"])
            if "random" in rng:
                random.setstate(rng["random"])

            start_episode = checkpoint["episode"] + 1
            best_reward = checkpoint.get("best_reward", -float("inf"))
            history = checkpoint.get("history", [])
            print(
                f"--> Successfully resumed from episode {checkpoint['episode']}. "
                f"Next episode: {start_episode}. Best reward so far: {best_reward:.3f}"
            )
        else:
            print(f"--> [Warning] Checkpoint '{args.resume}' not found. Starting from episode 1.")

    buffer = RolloutBuffer()
    max_steps_per_episode = env.num_routes * env.max_route_length + env.num_routes  # generous cap

    os.makedirs(args.checkpoint_dir, exist_ok=True)
    start_wall_time = time.time()
    max_seconds = args.max_hours * 3600.0 if args.max_hours > 0 else float("inf")

    last_ep_reward = 0.0

    for episode in range(start_episode, args.num_episodes + 1):
        # Time guard: exit cleanly before Kaggle's 12-hour session kill
        elapsed_total = time.time() - start_wall_time
        if elapsed_total >= max_seconds:
            print(f"\n[Time Guard] Elapsed time {elapsed_total / 3600.0:.2f}h reached safety limit ({args.max_hours:.2f}h).")
            print(f"[Time Guard] Saving checkpoint at completed episode {episode - 1} and exiting cleanly.")
            save_checkpoint(
                os.path.join(args.checkpoint_dir, "checkpoint_latest.pt"),
                episode - 1,
                agent,
                last_ep_reward,
                best_reward,
                history,
                args,
            )
            break

        t0 = time.time()
        buffer.clear()
        ep_reward, steps, last_obs = run_episode(env, agent, buffer, max_steps_per_episode)
        last_ep_reward = ep_reward

        # Bootstrap value for the final state (0 if truly terminal)
        with torch.no_grad():
            if steps >= max_steps_per_episode:
                _, _, last_value, _ = agent.act(last_obs)
            else:
                last_value = 0.0

        stats = agent.update(buffer, last_value)
        dt = time.time() - t0

        record = {
            "episode": episode,
            "steps": steps,
            "reward": float(ep_reward),
            "policy_loss": float(stats["policy_loss"]),
            "value_loss": float(stats["value_loss"]),
            "entropy": float(stats["entropy"]),
            "duration_s": float(dt),
        }
        history.append(record)

        if episode % args.log_every == 0:
            print(
                f"episode {episode:4d}/{args.num_episodes} | steps {steps:3d} | reward {ep_reward:8.3f} | "
                f"policy_loss {stats['policy_loss']:.4f} | value_loss {stats['value_loss']:.4f} | "
                f"entropy {stats['entropy']:.4f} | {dt:.1f}s"
            )

        # 1. Always save latest checkpoint (guards against mid-run crashes)
        save_checkpoint(
            os.path.join(args.checkpoint_dir, "checkpoint_latest.pt"),
            episode,
            agent,
            ep_reward,
            best_reward,
            history,
            args,
        )

        # 2. Save periodic checkpoint
        if episode % args.save_every == 0:
            save_checkpoint(
                os.path.join(args.checkpoint_dir, f"checkpoint_ep_{episode}.pt"),
                episode,
                agent,
                ep_reward,
                best_reward,
                history,
                args,
            )
            print(f"--> Saved periodic checkpoint: checkpoint_ep_{episode}.pt")

        # 3. Save best checkpoint
        if ep_reward > best_reward:
            best_reward = ep_reward
            save_checkpoint(
                os.path.join(args.checkpoint_dir, "checkpoint_best.pt"),
                episode,
                agent,
                ep_reward,
                best_reward,
                history,
                args,
            )
            print(f"--> New best reward: {best_reward:.3f}! Saved checkpoint_best.pt")

    print(f"\nTraining run complete. Checkpoints and training history saved to '{args.checkpoint_dir}'.")


if __name__ == "__main__":
    main()


Writing train.py


In [13]:
%%writefile visualize.py
"""
visualize.py - Publication-grade & interactive visualizations for the
Bloomington Transit Route Network Design Problem (TRNDP).
Replicates Figures 1 & 2 from Poudel & Li (arXiv:2512.19767).
"""
from __future__ import annotations

import json
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

TRANSIT_CENTER_NODE = "96"  # Highlighted in paper Figure 2


def load_network_for_vis(
    nodes_path: str = "./data/bloomington_nodes.csv",
    links_path: str = "./data/bloomington_links.csv",
    demand_path: str = "./data/bloomington_demand.csv",
    routes_path: str = "./data/bloomington_routes.json",
):
    nodes_df = pd.read_csv(nodes_path)
    links_df = pd.read_csv(links_path)
    demand_df = pd.read_csv(demand_path) if os.path.exists(demand_path) else None

    with open(routes_path, "r", encoding="utf-8") as f:
        routes_raw = json.load(f)

    routes_dict = {
        r["name"]: [str(n) for n in r["nodes"]]
        for r in routes_raw
    }
    node_xy = {str(row["name"]): (float(row["x"]), float(row["y"])) for _, row in nodes_df.iterrows()}
    return nodes_df, links_df, demand_df, routes_dict, node_xy


def plot_network_routes(
    routes_dict: dict[str, list[str]],
    node_xy: dict[str, tuple[float, float]],
    links_df: pd.DataFrame,
    title: str = "Transit Route Network",
    ax: plt.Axes | None = None,
    show_legend: bool = True,
    transit_center_id: str = TRANSIT_CENTER_NODE,
) -> plt.Axes:
    """Plots road network graph in light gray and overlaid transit routes in distinct colors."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(9, 9), dpi=150)

    # 1. Base street network (Definition 1)
    for _, link in links_df.iterrows():
        u, v = str(link["start"]), str(link["end"])
        if u in node_xy and v in node_xy:
            x0, y0 = node_xy[u]
            x1, y1 = node_xy[v]
            ax.plot([x0, x1], [y0, y1], color="#d0d0d0", linewidth=1.1, zorder=1)

    # 2. Base nodes
    xs = [node_xy[nid][0] for nid in node_xy]
    ys = [node_xy[nid][1] for nid in node_xy]
    ax.scatter(xs, ys, color="#aaaaaa", s=14, zorder=2, alpha=0.6)

    # 3. Palette for routes (using tab20 for up to 20 distinct high-contrast colors)
    color_map = plt.cm.tab20(np.linspace(0, 1, max(len(routes_dict), 1)))

    for (route_name, nodes), color in zip(routes_dict.items(), color_map):
        valid_nodes = [str(n) for n in nodes if str(n) in node_xy]
        if len(valid_nodes) < 2:
            continue
        rx = [node_xy[n][0] for n in valid_nodes]
        ry = [node_xy[n][1] for n in valid_nodes]
        ax.plot(rx, ry, color=color, linewidth=2.4, alpha=0.85, label=route_name, zorder=3)
        ax.scatter(rx, ry, color=color, s=22, zorder=4)

    # 4. Highlight Transit Center (Node 96) with a prominent marker (Figure 2)
    if transit_center_id in node_xy:
        tc_x, tc_y = node_xy[transit_center_id]
        ax.scatter(
            [tc_x], [tc_y],
            color="red",
            marker="*",
            s=280,
            edgecolor="black",
            linewidth=1.2,
            zorder=6,
            label=f"Transit Center ({transit_center_id})",
        )

    ax.set_title(title, fontsize=13, fontweight="bold", pad=10)
    ax.set_aspect("equal")
    ax.axis("off")
    if show_legend:
        ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False, fontsize=8, ncol=1)
    return ax


def plot_paper_comparison(
    real_routes: dict[str, list[str]],
    rl_routes: dict[str, list[str]],
    node_xy: dict[str, tuple[float, float]],
    links_df: pd.DataFrame,
    save_path: str = "figure2_comparison.png",
):
    """Replicates Figure 2 from Poudel & Li: Real-world vs. RL-designed routes side-by-side."""
    fig, axes = plt.subplots(1, 2, figsize=(18, 9), dpi=200)

    plot_network_routes(real_routes, node_xy, links_df, title="Real-world Baseline (Bloomington Transit)", ax=axes[0], show_legend=False)
    plot_network_routes(rl_routes, node_xy, links_df, title="RL-Designed Routes (GATv2 + PPO)", ax=axes[1], show_legend=False)

    fig.suptitle("Comparison of Real-World and RL-Designed Transit Route Networks\n(Bloomington, Indiana — 143 Nodes, 243 Edges)", fontsize=16, fontweight="bold", y=0.98)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, bbox_inches="tight", dpi=300)
        print(f"Saved side-by-side comparison figure to: {save_path}")
    plt.show()


def create_interactive_route_map(
    routes_dict: dict[str, list[str]],
    node_xy: dict[str, tuple[float, float]],
    links_df: pd.DataFrame,
    save_html_path: str = "transit_routes_interactive.html",
):
    """Creates an interactive Plotly map where routes can be toggled on/off in the browser."""
    import plotly.graph_objects as go

    fig = go.Figure()

    # 1. Add background street network edges
    edge_x, edge_y = [], []
    for _, link in links_df.iterrows():
        u, v = str(link["start"]), str(link["end"])
        if u in node_xy and v in node_xy:
            edge_x += [node_xy[u][0], node_xy[v][0], None]
            edge_y += [node_xy[u][1], node_xy[v][1], None]

    fig.add_trace(
        go.Scatter(
            x=edge_x, y=edge_y,
            line=dict(width=1.0, color="#d0d0d0"),
            hoverinfo="none",
            mode="lines",
            name="Road Network",
            showlegend=True,
        )
    )

    # 2. Add base nodes
    node_x = [node_xy[nid][0] for nid in node_xy]
    node_y = [node_xy[nid][1] for nid in node_xy]
    node_text = [f"Node: {nid}" for nid in node_xy]

    fig.add_trace(
        go.Scatter(
            x=node_x, y=node_y,
            mode="markers",
            marker=dict(size=4, color="#999999"),
            text=node_text,
            hoverinfo="text",
            name="Intersections / Stops",
        )
    )

    # 3. Add each transit route as a toggleable trace
    for route_name, nodes in routes_dict.items():
        valid_nodes = [str(n) for n in nodes if str(n) in node_xy]
        if len(valid_nodes) < 2:
            continue
        rx = [node_xy[n][0] for n in valid_nodes]
        ry = [node_xy[n][1] for n in valid_nodes]
        hover_labels = [f"{route_name}<br>Stop {i+1}: Node {n}" for i, n in enumerate(valid_nodes)]

        fig.add_trace(
            go.Scatter(
                x=rx, y=ry,
                mode="lines+markers",
                name=route_name,
                line=dict(width=3.5),
                marker=dict(size=7),
                text=hover_labels,
                hoverinfo="text",
            )
        )

    # 4. Highlight Transit Center (Node 96)
    if TRANSIT_CENTER_NODE in node_xy:
        tc_x, tc_y = node_xy[TRANSIT_CENTER_NODE]
        fig.add_trace(
            go.Scatter(
                x=[tc_x], y=[tc_y],
                mode="markers+text",
                marker=dict(size=14, color="red", symbol="star"),
                name="Transit Center (96)",
                text=["Transit Center (Node 96)"],
                textposition="top center",
                hoverinfo="text",
            )
        )

    fig.update_layout(
        title="Interactive Bloomington Transit Route Network (Click Legend to Toggle Routes)",
        template="plotly_white",
        showlegend=True,
        legend=dict(title="Toggle Routes:", yanchor="top", y=0.99, xanchor="left", x=1.02),
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, scaleanchor="x", scaleratio=1),
        width=950,
        height=750,
    )

    if save_html_path:
        fig.write_html(save_html_path)
        print(f"Interactive route map saved to: {save_html_path}")
    return fig


def plot_training_metrics(
    history_path: str = "./checkpoints/training_history.json",
    save_path: str = "training_metrics.png",
):
    """Plots episode rewards, policy loss, value loss, and entropy from training_history.json."""
    if not os.path.exists(history_path):
        print(f"No history file found at {history_path}. Run training first.")
        return

    with open(history_path, "r", encoding="utf-8") as f:
        history = json.load(f)

    if not history:
        print("History is empty.")
        return

    df = pd.DataFrame(history)
    fig, axes = plt.subplots(2, 2, figsize=(14, 9), dpi=150)

    # Reward
    axes[0, 0].plot(df["episode"], df["reward"], color="#1f77b4", linewidth=1.5)
    axes[0, 0].set_title("Episode Reward (Psi, Omega, Travel Time)", fontweight="bold")
    axes[0, 0].set_xlabel("Episode")
    axes[0, 0].set_ylabel("Reward")
    axes[0, 0].grid(True, linestyle="--", alpha=0.6)

    # Policy & Value Loss
    axes[0, 1].plot(df["episode"], df["policy_loss"], color="#ff7f0e", label="Policy Loss")
    axes[0, 1].plot(df["episode"], df["value_loss"], color="#2ca02c", label="Value Loss")
    axes[0, 1].set_title("Loss Trajectories", fontweight="bold")
    axes[0, 1].set_xlabel("Episode")
    axes[0, 1].set_ylabel("Loss")
    axes[0, 1].legend()
    axes[0, 1].grid(True, linestyle="--", alpha=0.6)

    # Policy Entropy
    axes[1, 0].plot(df["episode"], df["entropy"], color="#d62728", linewidth=1.5)
    axes[1, 0].set_title("Policy Entropy (Exploration)", fontweight="bold")
    axes[1, 0].set_xlabel("Episode")
    axes[1, 0].set_ylabel("Entropy")
    axes[1, 0].grid(True, linestyle="--", alpha=0.6)

    # Duration per episode
    axes[1, 1].plot(df["episode"], df["duration_s"], color="#9467bd", linewidth=1.5)
    axes[1, 1].set_title("Computation Time per Episode (seconds)", fontweight="bold")
    axes[1, 1].set_xlabel("Episode")
    axes[1, 1].set_ylabel("Seconds")
    axes[1, 1].grid(True, linestyle="--", alpha=0.6)

    plt.suptitle("PPO Training Progress & Convergence", fontsize=15, fontweight="bold", y=0.99)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, bbox_inches="tight", dpi=300)
        print(f"Training metrics plot saved to: {save_path}")
    plt.show()


def generate_routes_from_checkpoint(
    checkpoint_path: str = "./checkpoints/checkpoint_latest.pt",
    data_dir: str = "./data",
    num_routes: int = 16,
    max_route_length: int = 14,
    device: str = "cpu",
    seed: int = 0,
) -> dict[str, list[str]]:
    """
    Loads policy weights from a saved checkpoint, runs a fast rollout to build
    all routes, and returns the designed route dictionary without waiting
    for a full traffic simulation.
    """
    import torch
    from network_data import load_network_data
    from env import TransitEnv
    from models import ModelConfig
    from ppo_agent import PPOAgent, PPOConfig

    net = load_network_data(
        os.path.join(data_dir, "bloomington_nodes.csv"),
        os.path.join(data_dir, "bloomington_links.csv"),
        os.path.join(data_dir, "bloomington_demand.csv"),
        os.path.join(data_dir, "bloomington_routes.json"),
    )
    # Minimal t_sim (1s) since we only need the agent's route construction trajectory, not simulation metrics
    env = TransitEnv(net, num_routes=num_routes, max_route_length=max_route_length, t_sim=1.0, seed=seed)

    model_cfg = ModelConfig(
        node_feat_dim=16,
        edge_feat_dim=2,
        hidden_dim=128,
        num_heads=4,
        num_layers=4,
        num_routes=num_routes,
    )
    ppo_cfg = PPOConfig(device=device)
    agent = PPOAgent(model_cfg, ppo_cfg)

    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    agent.model.load_state_dict(checkpoint["model_state_dict"])
    agent.model.eval()

    obs, _ = env.reset(seed=seed)
    ep_done = False
    while not ep_done:
        with torch.no_grad():
            action, _, _, _ = agent.act(obs)
        obs, _, _, ep_done, _ = env.step(action)

    return env.completed_routes


Writing visualize.py


In [ ]:
# Download Bloomington TRNDP Dataset Directly via HTTPS (Bypasses Hugging Face library entirely)
import os
import json
import urllib.request

os.makedirs("./data", exist_ok=True)
BASE_URL = "https://huggingface.co/datasets/matrix-multiply/bloomington-tndp/resolve/main/standard"

files_to_download = {
    "bloomington_nodes_standard.csv": "./data/bloomington_nodes.csv",
    "bloomington_links_standard.csv": "./data/bloomington_links.csv",
    "bloomington_demand_standard.csv": "./data/bloomington_demand.csv",
    "bloomington_existing_routes.json": "./data/bloomington_routes.json",
}

all_exist = all(os.path.exists(path) for path in files_to_download.values())

if not all_exist:
    print("Downloading Bloomington network dataset directly via HTTPS...")
    for remote_name, local_path in files_to_download.items():
        url = f"{BASE_URL}/{remote_name}"
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req) as resp:
            data = resp.read()
        with open(local_path, "wb") as f:
            f.write(data)
        print(f"  ✓ Downloaded {remote_name} -> {local_path} ({len(data):,} bytes)")
    print("\nAll dataset files successfully saved locally to ./data/! Zero HF warnings.")
else:
    print("Local dataset files already exist in ./data/. Ready for offline training.")

# Verify files
try:
    import pandas as pd
    print(f"Nodes loaded        : {len(pd.read_csv('./data/bloomington_nodes.csv'))} stops/intersections")
    print(f"Links loaded        : {len(pd.read_csv('./data/bloomington_links.csv'))} road segments")
    print(f"Demand pairs loaded : {len(pd.read_csv('./data/bloomington_demand.csv'))} OD flows")
except ImportError:
    pass

with open("./data/bloomington_routes.json", "r", encoding="utf-8") as f:
    print(f"Existing routes     : {len(json.load(f))} routes")


In [ ]:
!python train.py --data_source csv --num_routes 2 --max_route_length 5 --t_sim 1000 --num_episodes 2 --rollout_steps 32 --save_every 1 --checkpoint_dir ./test_checkpoints

In [ ]:
# Verify checkpoint resume functionality works seamlessly
!python train.py --data_source csv --num_routes 2 --max_route_length 5 --t_sim 1000 --num_episodes 4 --rollout_steps 32 --save_every 1 --checkpoint_dir ./test_checkpoints --resume ./test_checkpoints/checkpoint_latest.pt

In [ ]:
# Full Training Run
# - --save_every 5: Saves periodic checkpoints every 5 episodes
# - --max_hours 11.0: Automatically saves and exits before Kaggle's 12-hour session kill
# - Checkpoints are continuously saved to ./checkpoints/checkpoint_latest.pt and checkpoint_best.pt
!python train.py --data_source csv --num_routes 16 --max_route_length 14 --t_sim 10000 --num_episodes 200 --save_every 5 --max_hours 11.0 --checkpoint_dir ./checkpoints --device cuda

In [ ]:
# How to Resume Across Kaggle Sessions:
# -----------------------------------------------------------------------------------------
# 1. When Session 1 finishes (after ~11 hours), its outputs (./checkpoints/checkpoint_latest.pt)
#    are saved in /kaggle/working/checkpoints/.
# 2. To start Session 2:
#    a. Create a Kaggle Dataset from Session 1's notebook output, OR
#    b. In your notebook, click 'Add Data' -> 'Notebook Output Files' -> select this notebook.
#    c. Run the command below, updating the path to point to the attached checkpoint:
#
# !python train.py --data_source csv --resume /kaggle/input/<previous-notebook-output>/checkpoints/checkpoint_latest.pt --num_episodes 200 --save_every 5 --max_hours 11.0 --device cuda


In [ ]:
# --- Visualizations ---
# Plot the Real-World Baseline Network (Figure 1 & 2 Style)
from visualize import load_network_for_vis, plot_network_routes, create_interactive_route_map
import matplotlib.pyplot as plt

nodes_df, links_df, demand_df, real_routes, node_xy = load_network_for_vis()

fig, ax = plt.subplots(figsize=(10, 10), dpi=150)
plot_network_routes(real_routes, node_xy, links_df, title="Real-World Bloomington Transit Network (16 Routes)", ax=ax)
plt.savefig("bloomington_real_world_network.png", bbox_inches="tight", dpi=300)
plt.show()


In [ ]:
# Create Interactive Plotly Route Map (Toggle routes on/off in the browser)
fig_interactive = create_interactive_route_map(real_routes, node_xy, links_df, save_html_path="bloomington_routes_interactive.html")
fig_interactive.show()


In [ ]:
# Plot Training Loss, Reward, and Entropy Convergence Curves
from visualize import plot_training_metrics
plot_training_metrics(history_path="./checkpoints/training_history.json", save_path="training_curves.png")


In [ ]:
# Replicate Figure 2 from Checkpoint: Real-World Baseline vs. RL-Designed Network
# (Can be run at ANY time, even if training was interrupted!)
import os
from visualize import load_network_for_vis, generate_routes_from_checkpoint, plot_paper_comparison, create_interactive_route_map

checkpoint_to_visualize = "./checkpoints/checkpoint_best.pt"
if not os.path.exists(checkpoint_to_visualize):
    checkpoint_to_visualize = "./checkpoints/checkpoint_latest.pt"

if os.path.exists(checkpoint_to_visualize):
    print(f"Loading checkpoint: {checkpoint_to_visualize}")
    nodes_df, links_df, _, real_routes, node_xy = load_network_for_vis()
    
    # Generate the 16 routes designed by the checkpoint policy (~2 seconds)
    rl_routes = generate_routes_from_checkpoint(checkpoint_to_visualize, device="cpu")
    print(f"Successfully generated {len(rl_routes)} transit routes from checkpoint policy!")
    
    # 1. Side-by-side comparison plot matching Figure 2 in Poudel & Li (2025)
    plot_paper_comparison(real_routes, rl_routes, node_xy, links_df, save_path="figure2_checkpoint_comparison.png")
    
    # 2. Interactive map for the RL-designed routes
    fig_rl_interactive = create_interactive_route_map(rl_routes, node_xy, links_df, save_html_path="rl_routes_interactive.html")
    fig_rl_interactive.show()
else:
    print("No checkpoint found yet at ./checkpoints/. Run at least 1 training episode first.")


In [ ]:
# Route Details & Fleet Sizing Analysis (Matching Paper Section IV-A & IX)
import pandas as pd
import json
from visualize import load_network_for_vis
from transit_assignment import build_route_geometries, assign_transit_demand, compute_frequencies
from network_data import load_network_data

net = load_network_data(
    "./data/bloomington_nodes.csv",
    "./data/bloomington_links.csv",
    "./data/bloomington_demand.csv",
    "./data/bloomington_routes.json"
)
nodes_df, links_df, _, routes_dict, node_xy = load_network_for_vis()

geometries = build_route_geometries(net, routes_dict)
segment_load = assign_transit_demand(net, geometries, alpha=0.3)
frequencies = compute_frequencies(geometries, segment_load)

route_summary = []
TRANSIT_CENTER = "96"

for r_name, nodes in routes_dict.items():
    geo = geometries.get(r_name)
    total_time_min = sum(geo.segment_time_s) / 60.0 if geo else 0.0
    
    # Calculate route length in km from link data
    total_dist_km = 0.0
    for u, v in zip(nodes[:-1], nodes[1:]):
        edge_match = links_df[
            ((links_df["start"].astype(str) == str(u)) & (links_df["end"].astype(str) == str(v))) |
            ((links_df["start"].astype(str) == str(v)) & (links_df["end"].astype(str) == str(u)))
        ]
        if not edge_match.empty:
            total_dist_km += edge_match.iloc[0]["length"] / 1000.0

    freq = frequencies.get(r_name, 1)
    headway_min = 60.0 / freq
    serves_tc = TRANSIT_CENTER in [str(n) for n in nodes]

    route_summary.append({
        "Route Name": r_name,
        "Stops (#)": len(nodes),
        "Length (km)": round(total_dist_km, 2),
        "Free-Flow Time (min)": round(total_time_min, 1),
        "Assigned Freq (bus/h)": freq,
        "Headway (min)": round(headway_min, 1),
        "Serves Transit Center (96)": "Yes" if serves_tc else "No"
    })

route_summary_df = pd.DataFrame(route_summary)
print(f"Total Routes: {len(route_summary_df)} | Total Hourly Bus Departures: {route_summary_df['Assigned Freq (bus/h)'].sum()} buses/h")
route_summary_df


In [ ]:
# Performance Comparison Dashboard (Replicating Table I from Poudel & Li, 2025)
import pandas as pd

table1_data = {
    "Category": ["Passenger", "Passenger", "Passenger", "Passenger", "Operator", "Operator", "Operator"],
    "Metric": [
        "Service Rate (%) ↑",
        "Wait Time (min) ↓",
        "Transfer Rate (%) ↓",
        "Travel Time (min) ↓",
        "Route Efficiency (pax/km) ↑",
        "Fleet Size ↓",
        "Bus Utilization (%) ↑"
    ],
    "α=0.3 Real-World": ["42.28", "14.19", "86.07", "48.12", "13.17", "89", "17.91"],
    "α=0.3 RL (Paper)": ["45.19 (+6.9%)", "10.32 (-27.3%)", "86.90 (+1.0%)", "48.82 (+1.5%)", "11.46 (-13.0%)", "92 (+3.4%)", "18.78 (+4.9%)"],
    "α=1.0 Real-World": ["58.20", "15.87", "82.14", "52.85", "62.74", "281", "32.15"],
    "α=1.0 RL (Paper)": ["73.10 (+25.6%)", "10.96 (-30.9%)", "88.26 (+7.4%)", "53.04 (+0.4%)", "72.06 (+14.9%)", "281 (0.0%)", "38.89 (+21.0%)"],
    "Our RL Model": ["(Run training)", "(Run training)", "(Run training)", "(Run training)", "(Run training)", "(Run training)", "(Run training)"]
}

metrics_df = pd.DataFrame(table1_data)

def style_categories(val):
    if val == "Passenger":
        return "background-color: #e8f4f8; font-weight: bold;"
    elif val == "Operator":
        return "background-color: #fbf0e4; font-weight: bold;"
    return ""

styled_table = metrics_df.style.map(style_categories, subset=["Category"])\
                               .set_properties(**{'text-align': 'center'})\
                               .set_table_styles([{'selector': 'th', 'props': [('text-align', 'center'), ('background-color', '#f2f2f2'), ('font-weight', 'bold')]}])
print("Summary Comparison: Real-World Baseline vs. RL-Designed Transit Network (Table I)")
styled_table
